**Importing dependencies**

In [2]:
pip install -r ./requirements.txt


In [4]:
import torch
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    DataCollatorWithPadding
)
from peft import get_peft_model, LoraConfig
from torch.utils.data import DataLoader
from torch.optim import AdamW
from tqdm.auto import tqdm

device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print(f"Using device: {device}")

Using device: cuda


**Base Model**

In [5]:
model_checkpoint = 'distilbert-base-uncased'
id2label = {0: "Negative", 1: "Positive"}
label2id = {"Negative": 0, "Positive": 1}

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, add_prefix_space=True)
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})

# Load Base Model
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint, num_labels=2, id2label=id2label, label2id=label2id
)
model.resize_token_embeddings(len(tokenizer))

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Embedding(30522, 768, padding_idx=0)

**Load data**

In [6]:
dataset = load_dataset('shawhin/imdb-truncated')
dataset

DatasetDict({
    train: Dataset({
        features: ['label', 'text'],
        num_rows: 1000
    })
    validation: Dataset({
        features: ['label', 'text'],
        num_rows: 1000
    })
})

**Preprocess data**

In [7]:
def tokenize_function(examples):
    # Tokenize and truncate text
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding=False # Padding will be handled dynamically by the collator later
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Remove raw text column and set format to PyTorch tensors (Crucial for manual loop)
tokenized_dataset = tokenized_dataset.remove_columns(["text"])
tokenized_dataset.set_format("torch")

# Data Collator (Handles dynamic padding)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Create DataLoaders
batch_size = 8 # Adjust based on your GPU memory
train_dataloader = DataLoader(
    tokenized_dataset["train"], shuffle=True, batch_size=batch_size, collate_fn=data_collator
)
eval_dataloader = DataLoader(
    tokenized_dataset["validation"], batch_size=batch_size, collate_fn=data_collator
)

# --- 5. SETUP PEFT (LoRA) ---
peft_config = LoraConfig(
    task_type="SEQ_CLS",
    r=4,
    lora_alpha=32,
    lora_dropout=0.01,
    target_modules=['q_lin']
)

model = get_peft_model(model, peft_config)
print("Trainable Parameters:")
model.print_trainable_parameters()
model.to(device)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Trainable Parameters:
trainable params: 628,994 || all params: 67,584,004 || trainable%: 0.9307


PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): DistilBertForSequenceClassification(
      (distilbert): DistilBertModel(
        (embeddings): Embeddings(
          (word_embeddings): Embedding(30522, 768, padding_idx=0)
          (position_embeddings): Embedding(512, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (transformer): Transformer(
          (layer): ModuleList(
            (0-5): 6 x TransformerBlock(
              (attention): DistilBertSdpaAttention(
                (dropout): Dropout(p=0.1, inplace=False)
                (q_lin): lora.Linear(
                  (base_layer): Linear(in_features=768, out_features=768, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.01, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=7

In [8]:
optimizer = AdamW(model.parameters(), lr=1e-3)
num_epochs = 10
accuracy_metric = evaluate.load("accuracy")

num_training_steps = num_epochs * len(train_dataloader)
progress_bar = tqdm(range(num_training_steps))

print("\nStarting Training...")

for epoch in range(num_epochs):
    # --- Training Phase ---
    model.train()
    total_loss = 0
    
    for batch in train_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        
        optimizer.step()
        optimizer.zero_grad()
        
        total_loss += loss.item()
        progress_bar.update(1)
        progress_bar.set_postfix(loss=loss.item())

    avg_train_loss = total_loss / len(train_dataloader)

    # --- Validation Phase ---
    model.eval()
    all_preds = []
    all_labels = []
    
    for batch in eval_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            outputs = model(**batch)
        
        logits = outputs.logits
        predictions = torch.argmax(logits, dim=-1)
        
        all_preds.extend(predictions.cpu().numpy())
        all_labels.extend(batch["labels"].cpu().numpy())

    eval_results = accuracy_metric.compute(predictions=all_preds, references=all_labels)
    print(f"Epoch {epoch + 1}/{num_epochs} | Train Loss: {avg_train_loss:.4f} | Val Accuracy: {eval_results['accuracy']:.4f}")

  0%|          | 0/1250 [00:00<?, ?it/s]


Starting Training...
Epoch 1/10 | Train Loss: 0.4810 | Val Accuracy: 0.8700
Epoch 2/10 | Train Loss: 0.2641 | Val Accuracy: 0.9070
Epoch 3/10 | Train Loss: 0.1738 | Val Accuracy: 0.8920
Epoch 4/10 | Train Loss: 0.1368 | Val Accuracy: 0.8790
Epoch 5/10 | Train Loss: 0.1104 | Val Accuracy: 0.8810
Epoch 6/10 | Train Loss: 0.1204 | Val Accuracy: 0.8730
Epoch 7/10 | Train Loss: 0.1361 | Val Accuracy: 0.8710
Epoch 8/10 | Train Loss: 0.1155 | Val Accuracy: 0.8660
Epoch 9/10 | Train Loss: 0.0699 | Val Accuracy: 0.8710
Epoch 10/10 | Train Loss: 0.0678 | Val Accuracy: 0.8700


**Evaluation**

In [11]:
# Save the adapter
output_dir = "distilbert-lora-manual"
model.save_pretrained(output_dir)
print(f"\nTraining Complete! Adapter saved to: {output_dir}")

print("\nRunning Inference on Test Sentences...")
text_list = [
    "It was good.", 
    "Not a fan, don't recommed.", 
    "Better than the first one.", 
    "Waste of time", 
    "This one is a pass."
]

model.eval() # Ensure model is in eval mode
for text in text_list:
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    
    predictions = torch.argmax(logits, dim=1).item()
    print(f"{text} -> {id2label[predictions]}")


Training Complete! Adapter saved to: distilbert-lora-manual

Running Inference on Test Sentences...
It was good. -> Positive
Not a fan, don't recommed. -> Negative
Better than the first one. -> Positive
Waste of time -> Negative
This one is a pass. -> Positive
